# 06 — Évaluation

Un RAG juridique doit être évalué séparément sur la récupération, la qualité des citations et la fidélité au contexte.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA = PROJECT_ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
INDEX = DATA / "index"
for directory in (RAW, PROCESSED, INDEX):
    directory.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT

In [ ]:
import pandas as pd

evaluation_path = DATA / "evaluation_questions.csv"
columns = [
    "question_id", "question", "language", "expected_source",
    "expected_page", "answerable", "notes"
]
if evaluation_path.exists():
    evaluation = pd.read_csv(evaluation_path)
else:
    evaluation = pd.DataFrame(columns=columns)
    evaluation.to_csv(evaluation_path, index=False)
evaluation

In [ ]:
def recall_at_k(retrieved: list[str], relevant: set[str], k: int) -> float:
    if not relevant:
        return 0.0
    return len(set(retrieved[:k]) & relevant) / len(relevant)

def reciprocal_rank(retrieved: list[str], relevant: set[str]) -> float:
    for rank, item in enumerate(retrieved, start=1):
        if item in relevant:
            return 1.0 / rank
    return 0.0

assert recall_at_k(["A", "B"], {"B"}, 2) == 1.0
assert reciprocal_rank(["A", "B"], {"B"}) == 0.5

## Grille qualitative

- la réponse est-elle entièrement soutenue par les extraits ?
- chaque affirmation importante possède-t-elle une citation correcte ?
- le système refuse-t-il de répondre lorsque la preuve manque ?
- la réponse conserve-t-elle le même sens en français, arabe et darija ?
- les anciennes versions d'un texte sont-elles clairement signalées ?